# 01 — Crossmatch

Crossmatch each QSO sample (DESI, W2M-current, W2M-legacy) to GALEX, PanSTARRS, UKIDSS,
AllWISE, and 2MASS using CDS XMatch, then combine the DESI+W2M outputs and generate
UV-excess candidate CSVs.

**Inputs:**
- `data/raw/COMBINED_QSOS_TAB.csv` (DESI, Fawcett+2023)
- `data/raw/FULL_W2M_SAMPLE_FIRST_VLASS.csv` (W2M, current)
- `data/raw/W2M_QSOs.csv` (W2M, legacy pipeline — kept for reproducibility)

**Outputs:**
- `data/matched/DESI_COMBINED_matched.csv`, `W2M_COMBINED_matched.csv`, `W2M_legacy_COMBINED_matched.csv`
- `data/matched/FINAL_COMBINED_QSOs.csv` (DESI + W2M-legacy), `FINAL_COMBINED_QSOs_W2M.csv` (DESI + W2M-current)
- `data/matched/uv_excess_candidates*.csv` (candidate lists; also consumed by `02_build_seds.ipynb` and visualized in `04_color_color.ipynb`)

**Script references:** `scripts/matching/*.py`

Run `/validate-crossmatch` after this notebook to check band coverage and flag contaminants before proceeding to notebook 02.

**Known issue:** always verify `SPECTYPE == 'QSO'` on the DESI sample before using it — the
base catalog is not pre-filtered by spectral type.</cell id="cell-0">


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import yaml
from astropy import units as u
from astropy.table import Table, join, unique, vstack
from astroquery.xmatch import XMatch

# notebooks live in <project>/notebooks/, so the project root is one level up
BASE_DIR = Path.cwd().parent
with open(BASE_DIR / "config" / "qso_params.yaml") as f:
    PARAMS = yaml.safe_load(f)

RADIUS = PARAMS["crossmatch"]["radius_arcsec"] * u.arcsec
MIN_CATALOG_MATCHES = PARAMS["crossmatch"]["min_catalog_matches"]

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_MATCHED = BASE_DIR / "data" / "matched"
DATA_MATCHED.mkdir(parents=True, exist_ok=True)

## DESI crossmatch

Source: `scripts/matching/desi_crossmatch_multi.py`. Matches the Fawcett+2023 DESI sample to
PanSTARRS and GALEX (inner join — both required), then UKIDSS/2MASS/AllWISE (left join — optional).

In [ ]:
DESI_CSV = DATA_RAW / "COMBINED_QSOS_TAB.csv"
DESI_OUT = DATA_MATCHED / "DESI_COMBINED_matched.csv"

DESI_BASE_COLS = ['TARGETID', 'RA', 'DEC', 'Z', 'SPECTYPE', 'EBV', 'EBV_ERR']
DESI_OUT_FIELDS = DESI_BASE_COLS + [
    'FUVmag', 'NUVmag',
    'gmag', 'rmag', 'imag', 'zmag', 'ymag',
    'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3',
    'Jmag', 'Hmag', 'Kmag',
    'W1mag', 'W2mag', 'W3mag', 'W4mag',
]

desi_base = Table.read(DESI_CSV, format='csv')
desi_base = desi_base[desi_base['SPECTYPE'] == 'QSO']  # never match stars/galaxies

# PanSTARRS and GALEX are required (inner join, keep closest match)
mt_ps1 = XMatch.query(cat1=desi_base, cat2="II/349/ps1", max_distance=RADIUS, colRA1='RA', colDec1='DEC')
mt_ps1.sort('angDist')
ps1 = unique(mt_ps1, keys='TARGETID', keep='first')
ps1 = ps1[['TARGETID', 'gmag', 'rmag', 'imag', 'zmag', 'ymag']]
step1 = join(desi_base, ps1, keys='TARGETID', join_type='inner')

mt_galex = XMatch.query(cat1=step1, cat2="II/335/galex_ais", max_distance=RADIUS, colRA1='RA', colDec1='DEC')
mt_galex.sort('angDist')
galex = unique(mt_galex, keys='TARGETID', keep='first')
galex = galex[['TARGETID', 'FUVmag', 'NUVmag']]
step2 = join(step1, galex, keys='TARGETID', join_type='inner')

# UKIDSS, 2MASS, AllWISE are optional (left join, preserves all PS1+GALEX sources)
mt_ukidss = XMatch.query(cat1=step2, cat2="II/319/las9", max_distance=RADIUS, colRA1='RA', colDec1='DEC')
mt_ukidss.sort('angDist')
ukidss = unique(mt_ukidss, keys='TARGETID', keep='first')
ukidss = ukidss[['TARGETID', 'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3']]

mt_2mass = XMatch.query(cat1=step2, cat2="II/246/out", max_distance=RADIUS, colRA1='RA', colDec1='DEC')
mt_2mass.sort('angDist')
mass2 = unique(mt_2mass, keys='TARGETID', keep='first')
mass2 = mass2[['TARGETID', 'Jmag', 'Hmag', 'Kmag']]

mt_allwise = XMatch.query(cat1=step2, cat2="II/328/allwise", max_distance=RADIUS, colRA1='RA', colDec1='DEC')
mt_allwise.sort('angDist')
allwise = unique(mt_allwise, keys='TARGETID', keep='first')
allwise = allwise[['TARGETID', 'W1mag', 'W2mag', 'W3mag', 'W4mag']]

desi_final = join(step2, ukidss, keys='TARGETID', join_type='left')
desi_final = join(desi_final, mass2, keys='TARGETID', join_type='left')
desi_final = join(desi_final, allwise, keys='TARGETID', join_type='left')

desi_final[DESI_OUT_FIELDS].write(DESI_OUT, format='csv', overwrite=True)
print(f"  -> {len(desi_final)} DESI sources written to {DESI_OUT}")

## W2M crossmatch (current)

Source: `scripts/matching/w2m_crossmatch_multi.py`. FIRST/VLASS radio-matched sample, restricted
to `spCl == 'redQSO'`; base catalog already includes SDSS ugriz, 2MASS, AllWISE, and E(B-V).
`gmag`/`rmag`/`imag`/`zmag` from SDSS (base) and PanSTARRS (matched) are disambiguated by
astropy's join suffixes: `_1` = SDSS, `_2` = PanSTARRS.

In [ ]:
W2M_CSV = DATA_RAW / "FULL_W2M_SAMPLE_FIRST_VLASS.csv"
W2M_OUT = DATA_MATCHED / "W2M_COMBINED_matched.csv"

W2M_OUT_FIELDS = [
    'designation', 'ra', 'dec', 'zsp', 'spCl', 'ebv',
    'umag', 'gmag_1', 'rmag_1', 'imag_1', 'zmag_1',
    'jmag', 'hmag', 'kmag',
    'w1mag', 'w2mag', 'w3mag', 'w4mag',
    'FIRST_peak', 'FIRST_int', 'VLASS_peak', 'VLASS_int',
    'broad', 'src',
    'FUVmag', 'NUVmag',
    'gmag_2', 'rmag_2', 'imag_2', 'zmag_2', 'ymag',
    'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3',
]

w2m_base = Table.read(W2M_CSV, format='csv')
w2m_base = w2m_base[w2m_base['spCl'] == 'redQSO']

mt_ps1 = XMatch.query(cat1=w2m_base, cat2="II/349/ps1", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_ps1.sort('angDist')
ps1 = unique(mt_ps1, keys='designation', keep='first')
ps1 = ps1[['designation', 'gmag', 'rmag', 'imag', 'zmag', 'ymag']]
step1 = join(w2m_base, ps1, keys='designation', join_type='inner')

mt_galex = XMatch.query(cat1=step1, cat2="II/335/galex_ais", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_galex.sort('angDist')
galex = unique(mt_galex, keys='designation', keep='first')
galex = galex[['designation', 'FUVmag', 'NUVmag']]
step2 = join(step1, galex, keys='designation', join_type='inner')

mt_ukidss = XMatch.query(cat1=step2, cat2="II/319/las9", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_ukidss.sort('angDist')
ukidss = unique(mt_ukidss, keys='designation', keep='first')
ukidss = ukidss[['designation', 'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3']]
w2m_final = join(step2, ukidss, keys='designation', join_type='left')

w2m_final[W2M_OUT_FIELDS].write(W2M_OUT, format='csv', overwrite=True)
print(f"  -> {len(w2m_final)} W2M sources written to {W2M_OUT}")

## W2M crossmatch (legacy)

Source: `scripts/matching/w2m_legacy_crossmatch_multi.py`. Pre-VLASS pipeline, superseded by the
current W2M crossmatch above; kept for reproducibility. Base catalog already includes SDSS ugriz
and AllWISE/2MASS photometry.

In [ ]:
W2M_LEGACY_CSV = DATA_RAW / "W2M_QSOs.csv"
W2M_LEGACY_OUT = DATA_MATCHED / "W2M_legacy_COMBINED_matched.csv"

W2M_LEGACY_OUT_FIELDS = [
    'designation', 'ra', 'dec', 'zsp',
    'umag', 'gmag_1', 'rmag_1', 'imag_1', 'zmag_1',
    'j_m_2mass', 'h_m_2mass', 'k_m_2mass',
    'w1mpro', 'w2mpro', 'w3mpro', 'w4mpro',
    'broad', 'src',
    'FUVmag', 'NUVmag',
    'gmag_2', 'rmag_2', 'imag_2', 'zmag_2', 'ymag',
    'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3',
]

w2m_legacy_base = Table.read(W2M_LEGACY_CSV, format='csv')

mt_ps1 = XMatch.query(cat1=w2m_legacy_base, cat2="II/349/ps1", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_ps1.sort('angDist')
ps1 = unique(mt_ps1, keys='designation', keep='first')
ps1 = ps1[['designation', 'gmag', 'rmag', 'imag', 'zmag', 'ymag']]
step1 = join(w2m_legacy_base, ps1, keys='designation', join_type='inner')

mt_galex = XMatch.query(cat1=step1, cat2="II/335/galex_ais", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_galex.sort('angDist')
galex = unique(mt_galex, keys='designation', keep='first')
galex = galex[['designation', 'FUVmag', 'NUVmag']]
step2 = join(step1, galex, keys='designation', join_type='inner')

mt_ukidss = XMatch.query(cat1=step2, cat2="II/319/las9", max_distance=RADIUS, colRA1='ra', colDec1='dec')
mt_ukidss.sort('angDist')
ukidss = unique(mt_ukidss, keys='designation', keep='first')
ukidss = ukidss[['designation', 'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3']]
w2m_legacy_final = join(step2, ukidss, keys='designation', join_type='left')

w2m_legacy_final[W2M_LEGACY_OUT_FIELDS].write(W2M_LEGACY_OUT, format='csv', overwrite=True)
print(f"  -> {len(w2m_legacy_final)} W2M-legacy sources written to {W2M_LEGACY_OUT}")

## Combine DESI + W2M

Source: `scripts/matching/combine_catalogs.py` (DESI + W2M-legacy) and `combine_catalogs_w2m.py`
(DESI + W2M-current). Stacks the two matched tables (outer join on columns) and drops duplicate
rows using GALEX/UKIDSS magnitude columns as the uniqueness key.

In [ ]:
DEDUPE_KEYS = ['FUVmag', 'NUVmag', 'ymag', 'yAperMag3', 'j_1AperMag3', 'hAperMag3', 'kAperMag3']


def combine_and_dedupe(desi_csv, w2m_csv, out_csv):
    desi = Table.read(desi_csv, format='csv')
    w2m = Table.read(w2m_csv, format='csv')

    combined = vstack([desi, w2m], join_type='outer')
    n_before = len(combined)

    # unique() requires no masked values; fill with 0.0 temporarily,
    # then recover original (unmasked) rows by index to avoid propagating fills
    copy = combined.filled(0.0)
    copy['_copy'] = np.arange(len(copy))
    copy = unique(copy, keys=DEDUPE_KEYS)
    combined = combined[copy['_copy']]

    n_dupes = n_before - len(combined)
    combined.write(out_csv, format='csv', overwrite=True)
    print(f"  -> {len(combined)} rows written to {out_csv} ({n_dupes} duplicates removed)")
    return combined


final_legacy = combine_and_dedupe(DESI_OUT, W2M_LEGACY_OUT, DATA_MATCHED / "FINAL_COMBINED_QSOs.csv")
final_w2m = combine_and_dedupe(DESI_OUT, W2M_OUT, DATA_MATCHED / "FINAL_COMBINED_QSOs_W2M.csv")

## UV-excess candidate selection

Source: consolidates `scripts/matching/candidates_to_csv_w2m.py`, `candidates_to_csv_w2m_gri.py`,
and `scripts/colorcolor/candidates_to_csv.py` (previously three near-identical copies of the same
`uv_excess_mask` logic). Applies the UV-excess criterion — FUV > NUV AND (NUV/G upturn OR
FUV/NUV upturn) AND E(B-V) > threshold — to DESI and W2M (current) separately, since W2M's E(B-V)
runs much lower (median ~0.03) and the E(B-V) cut is only applied to DESI. Also provides a
`g/r`-`r/i` (GRI) variant of the criterion.

This selection is the canonical source of the UV-excess candidate list; `04_color_color.ipynb`
visualizes but does not recompute it. Re-run this cell after any change to the crossmatch outputs
above.

In [ ]:
from synphot import units as su

LAM_FUV = PARAMS["band_wavelengths_AA"]["FUV"] * u.AA
LAM_NUV = PARAMS["band_wavelengths_AA"]["NUV"] * u.AA
LAM_G = PARAMS["band_wavelengths_AA"]["g"] * u.AA
LAM_R = PARAMS["band_wavelengths_AA"]["r"] * u.AA
LAM_I = PARAMS["band_wavelengths_AA"]["i"] * u.AA
FLAM_MAX_GALEX_PS1 = PARAMS["flux_outlier_thresholds_flam"]["galex_panstarrs"] * su.FLAM
EBV_MIN = PARAMS["uv_excess"]["ebv_min"]
TARGET_SAMPLE_MAX = PARAMS["uv_excess"]["target_sample_max"]


def mag_arr(col):
    if hasattr(col, 'filled'):
        return np.array(col.filled(np.nan), dtype=float)
    arr = np.array(col, dtype=str)
    arr[arr == '--'] = 'nan'
    return arr.astype(float)


def _flam(tbl, col, lam):
    flam = (mag_arr(tbl[col]) * u.ABmag).to(su.FLAM, u.spectral_density(lam))
    flam[flam > FLAM_MAX_GALEX_PS1] = np.nan
    return flam


def uv_excess_mask(tbl, gmag_col, rmag_col, imag_col=None, require_ebv=True, ebv_col='EBV'):
    """UV-excess criterion: FUV > NUV AND (NUV/G upturn OR FUV/NUV upturn),
    optionally AND E(B-V) > EBV_MIN. If imag_col is given, also includes the
    GRI-shifted analogue of the upturn signature (g/r > 1 & r/i < 1)."""
    flam_fuv = _flam(tbl, 'FUVmag', LAM_FUV)
    flam_nuv = _flam(tbl, 'NUVmag', LAM_NUV)
    flam_g = _flam(tbl, gmag_col, LAM_G)
    flam_r = _flam(tbl, rmag_col, LAM_R)

    f_fn = flam_fuv / flam_nuv
    f_ng = flam_nuv / flam_g
    f_gr = flam_g / flam_r

    flux_criterion = (
        ((f_ng > 1) & (f_gr < 1)) |
        ((f_fn > 1) & (f_ng < 1))
    )

    if imag_col is not None:
        flam_i = _flam(tbl, imag_col, LAM_I)
        f_ri = flam_r / flam_i
        flux_criterion = flux_criterion | ((f_gr > 1) & (f_ri < 1))

    if require_ebv:
        ebv = mag_arr(tbl[ebv_col])
        return flux_criterion & (ebv > EBV_MIN)
    return flux_criterion


def make_candidates(desi_csv, w2m_csv, out_csv, w2m_gmag='gmag_2', w2m_rmag='rmag_2',
                     w2m_imag=None, desi_imag=None):
    desi = Table.read(desi_csv, format='csv')
    desi_mask = uv_excess_mask(desi, gmag_col='gmag', rmag_col='rmag', imag_col=desi_imag, require_ebv=True)
    desi_cands = desi[desi_mask]

    # W2M sample is pre-restricted to spCl == 'redQSO' at crossmatch time; its EBV
    # runs much lower than DESI's (median ~0.03), so the E(B-V) cut is not applied.
    w2m = Table.read(w2m_csv, format='csv')
    w2m_mask = uv_excess_mask(w2m, gmag_col=w2m_gmag, rmag_col=w2m_rmag, imag_col=w2m_imag, require_ebv=False)
    w2m_cands = w2m[w2m_mask]

    candidates = vstack([desi_cands, w2m_cands], join_type='outer')
    candidates.write(out_csv, format='csv', overwrite=True)
    print(f"  -> {len(desi_cands)} DESI + {len(w2m_cands)} W2M = {len(candidates)} candidates written to {out_csv}")
    if len(candidates) > TARGET_SAMPLE_MAX:
        print(f"  WARNING: candidate sample ({len(candidates)}) exceeds target max ({TARGET_SAMPLE_MAX}) — investigate.")
    return candidates


# Standard (NUV/G, FUV/NUV) criterion, DESI + W2M-current
uv_cands_w2m = make_candidates(DESI_OUT, W2M_OUT, DATA_MATCHED / "uv_excess_candidates_w2m.csv")

# GRI-extended criterion, DESI + W2M-current
uv_cands_w2m_gri = make_candidates(
    DESI_OUT, W2M_OUT, DATA_MATCHED / "uv_excess_candidates_w2m_gri.csv",
    w2m_imag='imag_2', desi_imag='imag',
)

# Standard criterion, DESI + W2M-legacy
uv_cands_legacy = make_candidates(DESI_OUT, W2M_LEGACY_OUT, DATA_MATCHED / "uv_excess_candidates.csv")